In [0]:
%run ../log/00_audit_logging

In [0]:
#reads raw JSON files and appends them into Bronze
from datetime import datetime
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField, StringType, LongType, BooleanType, ArrayType
)

# Notebook Widgets
dbutils.widgets.text("load_type", "full", "Load Type (full/incremental)")
dbutils.widgets.text("batch_id", datetime.now().strftime("%Y%m%d_%H%M%S"), "Batch ID")

load_type = dbutils.widgets.get("load_type")
batch_id  = dbutils.widgets.get("batch_id")

start_time = datetime.now()

try:
    # Explicit Schemas (No inferSchema)
    full_load_envelope_schema = StructType([
        StructField("pulled_at",       StringType(), True),
        StructField("source_endpoint", StringType(), True),
        StructField("year_range",      StringType(), True),
        StructField("enriched",        BooleanType(), True),
        StructField("enrich_limit",    LongType(), True),
        StructField("record_count",    LongType(), True),
        StructField("results",         ArrayType(StringType()), True),
        StructField("_rescued_data",   StringType(), True)
    ])

    # Read ALL full load JSON files using wildcard *.json
    full_load_folder = "/Volumes/workspace/tmdb/raw/full_load/*.json"
    
    df_full_raw = (
        spark.read
        .schema(full_load_envelope_schema)
        .option("multiLine", "true")
        .option("mode", "PERMISSIVE")
        .option("columnNameOfCorruptRecord", "_rescued_data")
        .json(full_load_folder)
    )

    df_full_exploded = (
        df_full_raw
        .select(
            F.col("pulled_at"),
            F.col("source_endpoint"),
            F.col("year_range"),
            F.explode(F.col("results")).alias("raw_record")
        )
        .withColumn("source_collection", F.lit("full_load"))
    )

    # Union & add audit metadata
    common_cols = ["pulled_at", "source_endpoint", "source_collection", "raw_record"]
    
    df_bronze = (
        df_full_exploded.select(*common_cols)
        .withColumn("load_timestamp", F.current_timestamp())
        .withColumn("batch_id", F.lit(batch_id))
    )

    # Write to Bronze Table
    df_bronze.write.format("delta").mode("append").option("mergeSchema", "true").saveAsTable("workspace.tmdb.bronze_movies")
    
    record_count = df_bronze.count()
    end_time = datetime.now()
    log_pipeline_execution("Bronze", load_type, batch_id, start_time, end_time, "SUCCESS", record_count)

except Exception as e:
    end_time = datetime.now()
    log_pipeline_execution("Bronze", load_type, batch_id, start_time, end_time, "FAILURE", 0, str(e))
    raise e